# Nemotron Reasoning — Local GTX dev notebook

**Purpose.** GTX cards (6–12 GB VRAM, Pascal/Turing, no bf16 tensor cores) cannot host Nemotron-3-Nano-30B (30B MoE, ~60 GB bf16 / ~15 GB 4-bit weights + activations). This notebook instead lets you iterate on **data pipeline + training recipe locally** using a **small proxy model (Qwen2.5-1.5B-Instruct, ~3 GB in 4-bit)**. Once the recipe looks good locally, upload the data to Colab A100 and run `colab_workflow_v2.ipynb` to produce the real submittable adapter.

**The proxy adapter is NOT submittable to Kaggle** — Kaggle requires the Nemotron-3-Nano-30B-A3B base. Use this notebook to:

1. Generate + inspect synthetic training data
2. Optionally call the OpenAI API to produce verified CoT on the real train.csv
3. Run a short LoRA SFT on Qwen2.5-1.5B to sanity-check the training pipeline + loss curves
4. Run local accuracy eval against a held-out slice of train.csv
5. Ship the same `data/train_sft.jsonl` to Colab for the 30B run

**Hardware:** NVIDIA GTX 1060/1070/1080/1660/1660Ti/1660Super (6–12 GB VRAM). For Pascal (GTX 10xx) the script uses `fp16` (no bf16).

## 0. Detect GPU + set dtypes

In [ ]:
import os, sys, subprocess, platform
from pathlib import Path

WORK_ROOT = Path.cwd().resolve()
# If this notebook lives in kaggle/ inside the repo, project root is one up.
if (WORK_ROOT / "scripts").is_dir():
    PROJECT = WORK_ROOT
elif (WORK_ROOT.parent / "scripts").is_dir():
    PROJECT = WORK_ROOT.parent
else:
    PROJECT = WORK_ROOT
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Project root:", PROJECT)

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        n = torch.cuda.get_device_name(0)
        vram_gib = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        cc = torch.cuda.get_device_capability(0)
        HAS_BF16 = cc[0] >= 8  # Ampere+
        print(f"GPU: {n} ({vram_gib:.1f} GB, CC={cc[0]}.{cc[1]}, bf16={HAS_BF16})")
    else:
        HAS_BF16 = False; vram_gib = 0
        print("No CUDA GPU detected — will run CPU data prep only.")
except ImportError:
    HAS_BF16 = False; vram_gib = 0
    print("torch not yet installed; install in next cell.")

DTYPE = "bf16" if HAS_BF16 else "fp16"

## 1. Install deps (CPU wheels for torch if you have no CUDA; otherwise pick your CUDA wheel)

If torch is already installed with CUDA, skip the torch install line. Adjust `cu121` / `cu118` to match your installed CUDA runtime (`nvidia-smi` top-right).

In [ ]:
def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

# Uncomment ONE of these if torch is not installed yet:
# pip_install("--index-url", "https://download.pytorch.org/whl/cu121", "torch==2.4.*")
# pip_install("--index-url", "https://download.pytorch.org/whl/cu118", "torch==2.4.*")

pip_install(
    "transformers>=4.45,<5",
    "peft>=0.12",
    "trl>=0.12",
    "datasets",
    "accelerate>=0.34",
    "bitsandbytes>=0.43",
    "sentencepiece",
    "pandas", "numpy", "tqdm", "scikit-learn",
    "huggingface_hub",
)
for p in ("torch", "transformers", "peft", "trl", "bitsandbytes", "datasets"):
    try: print(f"  {p}: {__import__(p).__version__}")
    except Exception as e: print(f"  {p}: MISSING ({e})")

## 2. Put `train.csv` (Kaggle competition data) under `data/`

In [ ]:
DATA_DIR = PROJECT / "data"
(DATA_DIR / "reports").mkdir(parents=True, exist_ok=True)
(DATA_DIR / "synthetic").mkdir(parents=True, exist_ok=True)
if not (DATA_DIR / "train.csv").is_file():
    print(f"!! copy the competition train.csv into: {DATA_DIR / 'train.csv'}")
else:
    print("train.csv OK:", (DATA_DIR / "train.csv").stat().st_size, "bytes")

## 3. Phase 1 — EDA (CPU). Tokenizer = Qwen2.5-1.5B-Instruct (fast download ~3 GB)

We only use this tokenizer for length stats. The adapter's chat template is still produced against the proxy model later. When you move data to Colab, 02_prepare_data.py there will re-tokenize with the Nemotron tokenizer.

In [ ]:
PROXY_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
subprocess.run([sys.executable, "scripts/01_eda.py",
                "--data-dir", str(DATA_DIR),
                "--report-dir", str(DATA_DIR / "reports"),
                "--tokenizer-model", PROXY_MODEL], check=True)
print("EDA report:", DATA_DIR / "reports" / "eda_report.md")

## 4. Phase 2 — Build SFT data (synthetic + optional teacher CoT)

Set `OPENAI_API_KEY` to call GPT-4o-mini for verified chain-of-thought on the real train.csv. Costs a few dollars for ~500 rows; skip for a synthetic-only first pass.

In [ ]:
import getpass
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass.getpass("OpenAI key (blank = synthetic only): ")
SKIP_COT = not OPENAI_API_KEY
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

SYNTHETIC_PER_KIND = 300
MAX_PER_TYPE = 1200
LIMIT_TRAIN_FOR_COT = 300  # keep small on local; expand on Colab

cot_args = ["--skip-cot"] if SKIP_COT else [
    "--cot-backend", "openai", "--cot-model", "gpt-4o-mini",
    "--cot-max-tokens", "4096",
    "--limit-train", str(LIMIT_TRAIN_FOR_COT),
]
cmd = [sys.executable, "scripts/02_prepare_data.py",
       "--data-dir", str(DATA_DIR),
       "--synthetic-dir", str(DATA_DIR / "synthetic"),
       "--output", str(DATA_DIR / "train_sft.jsonl"),
       "--tokenizer-model", PROXY_MODEL,
       "--synthetic-per-kind", str(SYNTHETIC_PER_KIND),
       "--max-per-type", str(MAX_PER_TYPE),
       "--max-tokens-per-example", "4096"] + cot_args
print(" ".join(cmd)); subprocess.run(cmd, check=True)
print("SFT file:", (DATA_DIR / "train_sft.jsonl").stat().st_size, "bytes")

## 5. Phase 3 — Local LoRA SFT on proxy model (sanity-check recipe)

This trains **Qwen2.5-1.5B-Instruct + LoRA** using the same SFT pipeline. The resulting adapter is NOT submittable to Kaggle — it's a dev loop. If loss behaves correctly here (smooth decrease, eval_loss tracks train_loss, converges in ~2 epochs), the same hyperparameters are safe to run on Nemotron-30B in Colab.

Sized for a 6 GB GTX (1060/1660/1660 Ti/Super). Bump batch + seq length on 8–12 GB cards.

In [ ]:
LOCAL_MAX_SEQ = 1024 if vram_gib < 8 else 2048
LOCAL_BATCH = 1
LOCAL_GRAD_ACCUM = 16
LOCAL_EPOCHS = 2.0
LOCAL_LR = 2e-4

# Completion-only loss works well on small datasets; packing OFF for completion-only.
USE_COMPLETION_ONLY = True
USE_PACKING = False

train_env = os.environ.copy()
train_env["TOKENIZERS_PARALLELISM"] = "false"
train_env["NEMOTRON_KAGGLE_PATCHES"] = "0"
train_env["PYTHONUNBUFFERED"] = "1"
train_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
# Pascal (GTX 10xx) has no bf16 tensor cores — force fp16 at training level.
if not HAS_BF16:
    train_env["ACCELERATE_MIXED_PRECISION"] = "fp16"

cmd = [sys.executable, "scripts/03_train_lora.py",
       "--data-path", str(DATA_DIR / "train_sft.jsonl"),
       "--output-dir", "lora_adapter_proxy",
       "--checkpoint-dir", "lora_output_proxy",
       "--model-path", PROXY_MODEL,
       "--lora-target-mode", "hf_linear",   # q/k/v/o/up/down/gate, standard Llama-like targets
       "--lora-r", "32",
       "--lora-alpha", "64",
       "--lora-dropout", "0.05",
       "--batch-size", str(LOCAL_BATCH),
       "--grad-accum", str(LOCAL_GRAD_ACCUM),
       "--epochs", str(LOCAL_EPOCHS),
       "--lr", str(LOCAL_LR),
       "--max-seq-length", str(LOCAL_MAX_SEQ),
       "--warmup-ratio", "0.05",
       "--max-grad-norm", "1.0",
       "--neftune-alpha", "5.0",
       "--adapter-base-name", PROXY_MODEL,
       "--force-peft",
       "--no-nemotron-kaggle-patches",
       "--dataloader-workers", "0"]
if USE_PACKING:
    cmd.append("--packing")
if USE_COMPLETION_ONLY and not USE_PACKING:
    cmd.append("--completion-only")
print(" ".join(cmd), flush=True)

proc = subprocess.Popen(cmd, env=train_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout: print(line, end="", flush=True)
rc = proc.wait()
if rc != 0: raise subprocess.CalledProcessError(rc, cmd)

## 6. Local accuracy sanity-check (held-out slice of train.csv)

This runs greedy generation on a small held-out slice and computes exact-match vs gold. **Numbers will be lower than what you'll see on Kaggle** (proxy model is ~20x smaller than Nemotron-30B) — we're checking the *shape* of the learning curve + format compliance, not the absolute score.

In [ ]:
import json, pandas as pd, torch, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from scripts.utils.answer_extractor import answers_match, extract_boxed_answer
from scripts.utils.data_formatter import DEFAULT_SYSTEM_PROMPT

ADAPTER = PROJECT / "lora_adapter_proxy"
BASE = PROXY_MODEL
N_VAL = 100
MAX_NEW_TOKENS = 1024

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if HAS_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=bnb, device_map="auto", trust_remote_code=True,
)
model = PeftModel.from_pretrained(model, str(ADAPTER))
model.eval()

df = pd.read_csv(DATA_DIR / "train.csv")
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True).head(N_VAL)
correct = 0
rows = []
for i, r in df.iterrows():
    msgs = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT},
        {"role": "user", "content": str(r["prompt"])},
    ]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             temperature=None, top_p=None, pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
    pred = extract_boxed_answer(text)
    gold = str(r["answer"]).strip()
    ok = pred is not None and answers_match(gold, pred)
    correct += int(ok)
    rows.append({"id": r.get("id", i), "gold": gold, "pred": pred, "ok": ok})
    if i < 3:
        print(f"[{i}] gold={gold!r} pred={pred!r} ok={ok}")

acc = correct / len(df)
print(f"\nProxy accuracy: {acc:.3f} ({correct}/{len(df)})")
with (DATA_DIR / "reports" / "local_eval.jsonl").open("w") as f:
    for row in rows: f.write(json.dumps(row) + "\n")

## 7. Ship data → Colab for Nemotron-30B

The critical output of this notebook is `data/train_sft.jsonl`. Upload it (plus train.csv) to Colab and run `colab_workflow_v2.ipynb` — that notebook will skip the slow Phase 2 if `data/train_sft.jsonl` is already present. Or just let Colab regenerate with API CoT on the full train.csv.

In [ ]:
print("Files to upload to Colab for the 30B run:")
for p in [DATA_DIR / "train.csv", DATA_DIR / "train_sft.jsonl",
         DATA_DIR / "reports" / "eda_report.md"]:
    if p.is_file(): print("  ", p, p.stat().st_size, "bytes")